<a href="https://colab.research.google.com/github/ayyucedemirbas/ChIP-Seq/blob/main/ChIP_Seq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests, subprocess, os

# ENCODE API: GATA1 K562 aligned BAM files (hg19)
def get_encode_bam_urls(experiment_id):
    api_url = (
        f"https://www.encodeproject.org/search/"
        f"?type=File"
        f"&dataset=/experiments/{experiment_id}/"
        f"&file_format=bam"
        f"&output_type=alignments"
        f"&assembly=hg19"
        f"&format=json"
        f"&limit=all"
    )
    r = requests.get(api_url, headers={"Accept": "application/json"})
    files = r.json().get("@graph", [])
    return [
        {
            "accession": f["accession"],
            "url": f"https://www.encodeproject.org{f['href']}",
            "size_mb": round(f.get("file_size", 0) / 1e6, 1)
        }
        for f in files
    ]

# GATA1 K562 ChIP-seq
chip_experiments  = ["ENCSR000EFT", "ENCSR000EWM"]

ctrl_experiments  = ["ENCSR000EHM", "ENCSR000EWK"]

all_files = {}
for exp in chip_experiments + ctrl_experiments:
    bams = get_encode_bam_urls(exp)
    all_files[exp] = bams
    for b in bams:
        print(f"{exp}  {b['accession']}  {b['size_mb']} MB")

ENCSR000EFT  ENCFF942NUX  504.6 MB
ENCSR000EFT  ENCFF573VVJ  520.9 MB
ENCSR000EFT  ENCFF000YNG  495.2 MB
ENCSR000EFT  ENCFF000YNB  466.8 MB
ENCSR000EWM  ENCFF461UBM  135.0 MB
ENCSR000EWM  ENCFF000YMZ  308.0 MB
ENCSR000EWM  ENCFF000YNA  458.6 MB
ENCSR000EWM  ENCFF465DSP  288.0 MB
ENCSR000EHM  ENCFF000YQP  702.0 MB
ENCSR000EHM  ENCFF000YQO  652.1 MB
ENCSR000EHM  ENCFF732TLR  392.8 MB
ENCSR000EHM  ENCFF498SAC  427.4 MB
ENCSR000EWK  ENCFF000VEA  1701.4 MB
ENCSR000EWK  ENCFF051ZZR  1155.1 MB


In [ ]:
chip_files = {
    "gata1_snyder_rep1": "ENCFF000YNG",   # 495 MB — Snyder rep1
    "gata1_snyder_rep2": "ENCFF000YNB",   # 467 MB — Snyder rep2
    "gata1_farnham_rep1": "ENCFF000YMZ",  # 308 MB — Farnham rep1
    "gata1_farnham_rep2": "ENCFF000YNA",  # 459 MB — Farnham rep2
}

# Control samples
ctrl_files = {
    "ctrl_snyder":  "ENCFF000YQP",   # 702 MB — Snyder IgG
    "ctrl_farnham": "ENCFF051ZZR",   # 1155 MB — Farnham input
}


In [ ]:
os.makedirs("bam_files", exist_ok=True)

all_files = {**chip_files, **ctrl_files}

for name, acc in all_files.items():
    url = f"https://www.encodeproject.org/files/{acc}/@@download/{acc}.bam"
    out = f"bam_files/{name}.bam"
    print(f"Downloading: {name} ({acc})")
    subprocess.run(["wget", "-q", "--show-progress", "-O", out, url])
    subprocess.run(["wget", "-q", "-O", out + ".bai", url.replace(".bam", ".bam.bai")])
    print(f" {out}")

Downloading: gata1_snyder_rep1 (ENCFF000YNG)
 bam_files/gata1_snyder_rep1.bam
Downloading: gata1_snyder_rep2 (ENCFF000YNB)
 bam_files/gata1_snyder_rep2.bam
Downloading: gata1_farnham_rep1 (ENCFF000YMZ)
 bam_files/gata1_farnham_rep1.bam
Downloading: gata1_farnham_rep2 (ENCFF000YNA)
 bam_files/gata1_farnham_rep2.bam
Downloading: ctrl_snyder (ENCFF000YQP)
 bam_files/ctrl_snyder.bam
Downloading: ctrl_farnham (ENCFF051ZZR)
 bam_files/ctrl_farnham.bam


In [ ]:
!pip install pysam statsmodels

In [ ]:
import pysam
bam = pysam.AlignmentFile("bam_files/gata1_snyder_rep1.bam", "rb")
print([sq["SN"] for sq in bam.header["SQ"]][:5])
bam.close()

['chr1', 'chr2', 'chr3', 'chr4', 'chr5']


In [ ]:
!apt-get update
!apt-get install samtools

In [ ]:
import subprocess, os

bam_dir = "bam_files"

for f in os.listdir(bam_dir):
    if f.endswith(".bai"):
        os.remove(os.path.join(bam_dir, f))

for f in os.listdir(bam_dir):
    if f.endswith(".bam"):
        bam_path = os.path.join(bam_dir, f)
        print(f"Indexing: {f}")
        result = subprocess.run(
            ["samtools", "index", bam_path],
            capture_output=True, text=True
        )
        if result.returncode != 0:
            print(f"  Error: {result.stderr}")
        else:
            bai_size = os.path.getsize(bam_path + ".bai")
            print(f" {f}.bai — {bai_size:,} bytes")

In [ ]:
import os
bam_dir = "bam_files"
for f in sorted(os.listdir(bam_dir)):
    path = os.path.join(bam_dir, f)
    print(f"{f:45s}  {os.path.getsize(path):>12,} bytes")

ctrl_farnham.bam                               1,155,120,890 bytes
ctrl_farnham.bam.bai                              2,703,904 bytes
ctrl_snyder.bam                                 702,043,807 bytes
ctrl_snyder.bam.bai                               3,303,128 bytes
gata1_farnham_rep1.bam                          307,966,177 bytes
gata1_farnham_rep1.bam.bai                        4,286,552 bytes
gata1_farnham_rep2.bam                          458,637,798 bytes
gata1_farnham_rep2.bam.bai                        3,519,736 bytes
gata1_snyder_rep1.bam                           495,204,944 bytes
gata1_snyder_rep1.bam.bai                         3,010,720 bytes
gata1_snyder_rep2.bam                           466,807,479 bytes
gata1_snyder_rep2.bam.bai                         3,098,248 bytes


In [ ]:
for fname in os.listdir(bam_dir):
    if fname.endswith(".bai"):
        print(fname)

ctrl_snyder.bam.bai
gata1_snyder_rep2.bam.bai
gata1_farnham_rep1.bam.bai
gata1_farnham_rep2.bam.bai
ctrl_farnham.bam.bai
gata1_snyder_rep1.bam.bai


In [ ]:
import pysam
import numpy as np
import pandas as pd
from scipy import stats
from scipy.ndimage import gaussian_filter1d
from statsmodels.stats.multitest import multipletests
import os
import sys


def ensure_bam_index(bam_path):
    import subprocess
    index_path = bam_path + ".bai"
    if not os.path.exists(index_path):
        print(f"    Sorting {os.path.basename(bam_path)}...")
        sorted_path = bam_path.replace(".bam", "_sorted.bam")
        subprocess.run(["samtools", "sort", "-o", sorted_path, bam_path], check=True)
        os.replace(sorted_path, bam_path)
        print(f"    Indexing {os.path.basename(bam_path)}...")
        subprocess.run(["samtools", "index", bam_path], check=True)
        print(f"    Done.")


def get_chrom_names(bam_path):
    with pysam.AlignmentFile(bam_path, "rb") as bam:
        names = [sq["SN"] for sq in bam.header["SQ"]]
    return names


def resolve_chrom(bam_path, chrom):
    available = get_chrom_names(bam_path)
    if chrom in available:
        return chrom
    alt = chrom.replace("chr", "") if chrom.startswith("chr") else "chr" + chrom
    if alt in available:
        print(f"    Chromosome '{chrom}' not found, using '{alt}' instead.")
        return alt
    raise ValueError(f"Chromosome '{chrom}' not found in BAM. Available: {available[:10]}")


def compute_coverage(bam_path, chrom, chrom_length, extend_length=200):
    chrom = resolve_chrom(bam_path, chrom)
    coverage = np.zeros(chrom_length, dtype=np.float32)
    with pysam.AlignmentFile(bam_path, "rb", index_filename=bam_path + ".bai") as bam:
        for read in bam.fetch(chrom, 0, chrom_length):
            if read.is_unmapped or read.is_duplicate:
                continue
            if read.is_reverse:
                end = min(read.reference_end, chrom_length)
                start = max(end - extend_length, 0)
            else:
                start = read.reference_start
                end = min(start + extend_length, chrom_length)
            coverage[start:end] += 1
    return coverage


def normalize_coverage(coverage, bam_path, scale_factor=1e6):
    with pysam.AlignmentFile(bam_path, "rb", index_filename=bam_path + ".bai") as bam:
        total_reads = bam.mapped
    if total_reads == 0:
        return coverage
    return coverage * (scale_factor / total_reads)


def call_peaks(coverage, control_coverage, chrom, min_fold_change=2.0,
               min_reads=5, smooth_sigma=5, merge_gap=50):
    smoothed = gaussian_filter1d(coverage, sigma=smooth_sigma)
    smoothed_ctrl = gaussian_filter1d(control_coverage, sigma=smooth_sigma) + 1e-6
    fold_change = smoothed / smoothed_ctrl
    mask = (smoothed >= min_reads) & (fold_change >= min_fold_change)
    peaks = []
    in_peak = False
    peak_start = 0
    for i in range(len(mask)):
        if mask[i] and not in_peak:
            peak_start = i
            in_peak = True
        elif not mask[i] and in_peak:
            peaks.append({
                "chrom": chrom,
                "start": peak_start,
                "end": i,
                "width": i - peak_start,
                "max_coverage": float(smoothed[peak_start:i].max()),
                "mean_fold_change": float(fold_change[peak_start:i].mean())
            })
            in_peak = False
    if in_peak:
        peaks.append({
            "chrom": chrom,
            "start": peak_start,
            "end": len(mask),
            "width": len(mask) - peak_start,
            "max_coverage": float(smoothed[peak_start:].max()),
            "mean_fold_change": float(fold_change[peak_start:].mean())
        })
    merged = []
    for peak in sorted(peaks, key=lambda x: x["start"]):
        if merged and peak["start"] <= merged[-1]["end"] + merge_gap:
            merged[-1]["end"] = max(merged[-1]["end"], peak["end"])
            merged[-1]["width"] = merged[-1]["end"] - merged[-1]["start"]
            merged[-1]["max_coverage"] = max(merged[-1]["max_coverage"], peak["max_coverage"])
        else:
            merged.append(peak.copy())
    return merged


def write_bed(peaks, output_path):
    with open(output_path, "w") as f:
        for i, peak in enumerate(peaks):
            score = int(min(peak.get("max_coverage", 0) * 10, 1000))
            f.write(f"{peak['chrom']}\t{peak['start']}\t{peak['end']}\t"
                    f"peak_{i+1}\t{score}\t.\n")


def read_bed(bed_path):
    peaks = []
    with open(bed_path) as f:
        for line in f:
            if line.startswith("#") or not line.strip():
                continue
            parts = line.strip().split("\t")
            peaks.append({
                "chrom": parts[0],
                "start": int(parts[1]),
                "end": int(parts[2]),
                "name": parts[3] if len(parts) > 3 else "."
            })
    return peaks


def merge_consensus_peaks(all_peak_lists, merge_gap=100):
    all_peaks = []
    for peak_list in all_peak_lists:
        all_peaks.extend(peak_list)
    all_peaks.sort(key=lambda x: (x["chrom"], x["start"]))
    consensus = []
    for peak in all_peaks:
        if consensus and peak["chrom"] == consensus[-1]["chrom"] and \
                peak["start"] <= consensus[-1]["end"] + merge_gap:
            consensus[-1]["end"] = max(consensus[-1]["end"], peak["end"])
        else:
            consensus.append({"chrom": peak["chrom"],
                               "start": peak["start"],
                               "end": peak["end"]})
    return consensus


def count_reads_in_peaks(bam_path, consensus_peaks):
    counts = []
    with pysam.AlignmentFile(bam_path, "rb", index_filename=bam_path + ".bai") as bam:
        for peak in consensus_peaks:
            try:
                count = bam.count(peak["chrom"], peak["start"], peak["end"],
                                  read_callback=lambda r: not r.is_unmapped and not r.is_duplicate)
            except ValueError:
                count = 0
            counts.append(count)
    return np.array(counts, dtype=np.int32)


def build_count_matrix(bam_paths, consensus_peaks, sample_names):
    matrix = {}
    for bam_path, name in zip(bam_paths, sample_names):
        print(f"  Counting reads in peaks for sample: {name}")
        ensure_bam_index(bam_path)
        matrix[name] = count_reads_in_peaks(bam_path, consensus_peaks)
    return pd.DataFrame(matrix)


def tmm_normalize(count_matrix):
    totals = count_matrix.sum(axis=0)
    ref_idx = np.argmin(np.abs(totals - totals.mean()))
    ref = count_matrix.iloc[:, ref_idx].values.astype(float) + 0.5
    normalized = count_matrix.copy().astype(float)
    for col in count_matrix.columns:
        sample = count_matrix[col].values.astype(float) + 0.5
        log_ratio = np.log2(sample / ref)
        abs_log = np.abs(log_ratio)
        trim_cutoff = np.percentile(abs_log, 30)
        keep = abs_log <= trim_cutoff
        if keep.sum() < 5:
            scale = totals[col] / totals.iloc[ref_idx]
        else:
            weights = 1.0 / (1.0 / sample[keep] + 1.0 / ref[keep])
            scale = 2 ** np.average(log_ratio[keep], weights=weights)
        normalized[col] = count_matrix[col].values / scale
    return normalized


def differential_binding(count_matrix, group1_samples, group2_samples,
                          consensus_peaks, min_count=5):
    results = []
    g1 = count_matrix[group1_samples].values.astype(float)
    g2 = count_matrix[group2_samples].values.astype(float)
    for i, peak in enumerate(consensus_peaks):
        row_g1 = g1[i]
        row_g2 = g2[i]
        if row_g1.mean() < min_count and row_g2.mean() < min_count:
            results.append({
                "chrom": peak["chrom"], "start": peak["start"], "end": peak["end"],
                "mean_g1": row_g1.mean(), "mean_g2": row_g2.mean(),
                "log2fc": np.nan, "pvalue": np.nan, "significant": False
            })
            continue
        pseudo = 0.5
        log2fc = np.log2((row_g2.mean() + pseudo) / (row_g1.mean() + pseudo))
        _, pvalue = stats.ttest_ind(np.log2(row_g1 + pseudo), np.log2(row_g2 + pseudo))
        results.append({
            "chrom": peak["chrom"], "start": peak["start"], "end": peak["end"],
            "mean_g1": row_g1.mean(), "mean_g2": row_g2.mean(),
            "log2fc": log2fc, "pvalue": pvalue, "significant": False
        })
    df = pd.DataFrame(results)
    valid = df["pvalue"].notna()
    if valid.sum() > 0:
        _, fdr, _, _ = multipletests(df.loc[valid, "pvalue"].values, method="fdr_bh")
        df.loc[valid, "fdr"] = fdr
        df["significant"] = (df.get("fdr", pd.Series([1.0] * len(df))) < 0.05) & \
                             (df["log2fc"].abs() > 1.0)
    else:
        df["fdr"] = np.nan
    return df


def run_chipseq_pipeline(chip_bam_paths, control_bam_paths, sample_names,
                         group1_names, group2_names, chrom, chrom_length,
                         output_dir="chipseq_output", extend_length=200):
    os.makedirs(output_dir, exist_ok=True)

    print("Step 1: Computing and normalizing coverage")
    chip_coverages = []
    for bam_path, name in zip(chip_bam_paths, sample_names):
        print(f"  Processing {name}")
        ensure_bam_index(bam_path)
        cov = compute_coverage(bam_path, chrom, chrom_length, extend_length)
        cov_norm = normalize_coverage(cov, bam_path)
        chip_coverages.append(cov_norm)

    control_coverages = []
    for bam_path in control_bam_paths:
        ensure_bam_index(bam_path)
        cov = compute_coverage(bam_path, chrom, chrom_length, extend_length)
        cov_norm = normalize_coverage(cov, bam_path)
        control_coverages.append(cov_norm)

    mean_control = np.mean(control_coverages, axis=0)

    print("Step 2: Calling peaks per sample")
    all_peak_lists = []
    for cov, name in zip(chip_coverages, sample_names):
        peaks = call_peaks(cov, mean_control, chrom)
        print(f"  {name}: {len(peaks)} peaks called")
        bed_out = os.path.join(output_dir, f"{name}_peaks.bed")
        write_bed(peaks, bed_out)
        all_peak_lists.append(peaks)

    print("Step 3: Building consensus peak set")
    consensus_peaks = merge_consensus_peaks(all_peak_lists)
    print(f"  Total consensus peaks: {len(consensus_peaks)}")
    consensus_bed = os.path.join(output_dir, "consensus_peaks.bed")
    write_bed(consensus_peaks, consensus_bed)

    print("Step 4: Counting reads in consensus peaks")
    count_matrix = build_count_matrix(chip_bam_paths, consensus_peaks, sample_names)
    count_matrix.index = [f"{p['chrom']}:{p['start']}-{p['end']}" for p in consensus_peaks]
    raw_counts_path = os.path.join(output_dir, "raw_counts.csv")
    count_matrix.to_csv(raw_counts_path)

    print("Step 5: TMM normalization")
    norm_matrix = tmm_normalize(count_matrix)
    norm_counts_path = os.path.join(output_dir, "normalized_counts.csv")
    norm_matrix.to_csv(norm_counts_path)

    print("Step 6: Differential binding analysis")
    diff_results = differential_binding(norm_matrix, group1_names, group2_names, consensus_peaks)
    sig_count = diff_results["significant"].sum()
    print(f"  Significant differentially bound peaks: {sig_count}")
    results_path = os.path.join(output_dir, "differential_binding.csv")
    diff_results.to_csv(results_path, index=False)

    sig_peaks = diff_results[diff_results["significant"]].copy()
    sig_bed_path = os.path.join(output_dir, "significant_peaks.bed")
    write_bed(sig_peaks.rename(columns={"log2fc": "max_coverage"}).to_dict("records"), sig_bed_path)

    print(f"\nPipeline complete. Results saved to: {output_dir}/")
    return diff_results, count_matrix, consensus_peaks


def verify_bam_files(bam_dir, expected_files):
    missing = []
    no_index = []
    for fname in expected_files:
        path = os.path.join(bam_dir, fname)
        if not os.path.exists(path):
            missing.append(fname)
        elif not os.path.exists(path + ".bai"):
            no_index.append(fname)
    if missing:
        print(f"[ERROR] Missing BAM files: {missing}")
        sys.exit(1)
    if no_index:
        print(f"[WARNING] Missing index (.bai) for: {no_index}")
        print("  Run: samtools index <file.bam>  for each one")


if __name__ == "__main__":
    BAM_DIR = "bam_files"

    # ENCODE GATA1 K562 ChIP-seq
    # Group 1: Snyder lab (ENCSR000EFT) — ENCFF000YNG, ENCFF000YNB
    # Group 2: Farnham lab (ENCSR000EWM) — ENCFF000YMZ, ENCFF000YNA
    # Controls: Snyder IgG (ENCFF000YQP) + Farnham input (ENCFF051ZZR)

    chip_bams = [
        os.path.join(BAM_DIR, "gata1_snyder_rep1.bam"),   # ENCFF000YNG
        os.path.join(BAM_DIR, "gata1_snyder_rep2.bam"),   # ENCFF000YNB
        os.path.join(BAM_DIR, "gata1_farnham_rep1.bam"),  # ENCFF000YMZ
        os.path.join(BAM_DIR, "gata1_farnham_rep2.bam"),  # ENCFF000YNA
    ]
    control_bams = [
        os.path.join(BAM_DIR, "ctrl_snyder.bam"),         # ENCFF000YQP
        os.path.join(BAM_DIR, "ctrl_farnham.bam"),        # ENCFF051ZZR
    ]
    sample_names = [
        "snyder_rep1", "snyder_rep2",
        "farnham_rep1", "farnham_rep2",
    ]
    group1 = ["snyder_rep1", "snyder_rep2"]
    group2 = ["farnham_rep1", "farnham_rep2"]

    verify_bam_files(BAM_DIR, [os.path.basename(p) for p in chip_bams + control_bams])

    results, counts, peaks = run_chipseq_pipeline(
        chip_bam_paths=chip_bams,
        control_bam_paths=control_bams,
        sample_names=sample_names,
        group1_names=group1,
        group2_names=group2,
        chrom="chr19",
        chrom_length=58617616,
        output_dir="gata1_chipseq_output",
        extend_length=200
    )

    print("\nTop differentially bound peaks:")
    top = results.dropna(subset=["fdr"]).sort_values("fdr").head(10)
    print(top[["chrom", "start", "end", "log2fc", "pvalue", "fdr", "significant"]].to_string(index=False))

Step 1: Computing and normalizing coverage
  Processing snyder_rep1
  Processing snyder_rep2
  Processing farnham_rep1
  Processing farnham_rep2
Step 2: Calling peaks per sample
  snyder_rep1: 35 peaks called
  snyder_rep2: 67 peaks called
  farnham_rep1: 112 peaks called
  farnham_rep2: 37 peaks called
Step 3: Building consensus peak set
  Total consensus peaks: 177
Step 4: Counting reads in consensus peaks
  Counting reads in peaks for sample: snyder_rep1
  Counting reads in peaks for sample: snyder_rep2
  Counting reads in peaks for sample: farnham_rep1
  Counting reads in peaks for sample: farnham_rep2
Step 5: TMM normalization
Step 6: Differential binding analysis
  Significant differentially bound peaks: 0

Pipeline complete. Results saved to: gata1_chipseq_output/

Top differentially bound peaks:
chrom    start      end    log2fc   pvalue      fdr  significant
chr19   944215   944332  1.621425 0.010109 0.269362        False
chr19  2171888  2172106 -3.608023 0.037853 0.269362    

/usr/local/lib/python3.12/dist-packages/scipy/stats/_axis_nan_policy.py:579: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)
